# BirdCLEF 2026 — Perch v2 + Ridge Student (0.911 baseline)

Adapted from brucewu1200/birdclef-2026-cvlb-assets-0911

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Detect environment
ON_KAGGLE = Path('/kaggle/input').exists()

if ON_KAGGLE:
    # Debug: list all input directories
    input_dir = Path('/kaggle/input')
    print('Available input directories:')
    for d in sorted(input_dir.iterdir()):
        print(f'  {d.name}/')
        # Show top-level contents
        for f in sorted(d.iterdir())[:10]:
            suffix = '/' if f.is_dir() else f' ({f.stat().st_size:,} bytes)'
            print(f'    {f.name}{suffix}')
    
    # Find competition data (may be birdclef-2026 or competitions/birdclef-2026)
    BASE_DIR = None
    for candidate in [
        input_dir / 'birdclef-2026',
        input_dir / 'competitions' / 'birdclef-2026',
    ]:
        if (candidate / 'sample_submission.csv').exists():
            BASE_DIR = candidate
            break
    if BASE_DIR is None:
        # Search for sample_submission.csv anywhere in input
        for p in input_dir.rglob('sample_submission.csv'):
            BASE_DIR = p.parent
            break
    assert BASE_DIR is not None, f'Could not find sample_submission.csv under {input_dir}'
    
    # Find assets dataset
    ASSET_DIR = None
    for candidate in [
        input_dir / 'birdclef-2026-cvlb-assets-0911',
        input_dir / 'datasets' / 'brucewu1200' / 'birdclef-2026-cvlb-assets-0911',
    ]:
        if (candidate / 'clip_student_bundle.pkl').exists():
            ASSET_DIR = candidate
            break
    if ASSET_DIR is None:
        for p in input_dir.rglob('clip_student_bundle.pkl'):
            ASSET_DIR = p.parent
            break
    assert ASSET_DIR is not None, f'Could not find clip_student_bundle.pkl under {input_dir}'
    
    # Find Perch v2 model
    MODEL_DIR = None
    for candidate in [
        input_dir / 'bird-vocalization-classifier' / 'tensorflow2' / 'perch_v2_cpu' / '1',
        input_dir / 'models' / 'google' / 'bird-vocalization-classifier' / 'tensorflow2' / 'perch_v2_cpu' / '1',
    ]:
        if candidate.exists():
            MODEL_DIR = candidate
            break
    # MODEL_DIR can be None — we'll use ONNX instead
    
    WORK_DIR = Path('/kaggle/working')
    
    # Install ONLY onnxruntime from bundled wheels (skip numpy/scipy/etc to avoid conflicts)
    wheel_dir = ASSET_DIR / 'wheels'
    INSTALL_ONLY = {'onnxruntime', 'flatbuffers', 'protobuf', 'sympy', 'mpmath', 'packaging'}
    if wheel_dir.exists():
        for whl in sorted(wheel_dir.glob('*.whl')):
            pkg_name = whl.name.split('-')[0].lower().replace('_', '-')
            if pkg_name in INSTALL_ONLY or any(pkg_name.startswith(x) for x in INSTALL_ONLY):
                print(f'Installing {whl.name}...')
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '--quiet', str(whl)])
            else:
                print(f'Skipping {whl.name} (using Kaggle pre-installed version)')
        print('Wheels installed.')
else:
    # Local development paths
    ROOT = Path('/home/swatson/work/MachineLearning/kaggle/BirdCLEF')
    ASSET_DIR = ROOT / 'data' / 'external' / 'birdclef-0911'
    BASE_DIR = ROOT / 'data' / 'raw'
    WORK_DIR = ROOT / 'data' / 'processed' / 'perch_0911_output'
    MODEL_DIR = ROOT / 'perch_v2' / 'models' / 'perch_v2'
    WORK_DIR.mkdir(parents=True, exist_ok=True)

ONNX_PATH = ASSET_DIR / 'perch_v2_no_dft.onnx'
print(f'\nResolved paths:')
print(f'  ASSET_DIR: {ASSET_DIR}')
print(f'  BASE_DIR:  {BASE_DIR}')
print(f'  MODEL_DIR: {MODEL_DIR}')
print(f'  ONNX_PATH: {ONNX_PATH} (exists={ONNX_PATH.exists()})')
print(f'  WORK_DIR:  {WORK_DIR}')

In [ ]:
from __future__ import annotations

import gc
import json
import pickle
import re
import time
import importlib
from typing import Any

import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

TEXTURE_TAXA = {"Amphibia", "Insecta"}
FILENAME_RE = re.compile(r".+_(S\d+)_(\d{8})_(\d{6})$")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

In [ ]:
CLIP_CFG = {
    "fallback_transfer_weight": 0.9,
    "calibration_blend": 0.8,
    "soundscape_calib_alpha": 2.0,
    "soundscape_target_teacher_weight": 0.7,
    "soundscape_min_teacher_weight": 0.25,
    "min_calibration_pos": 3,
    "prior_weight_event": 0.4,
    "prior_weight_texture": 0.4,
    "site_shrink": 8.0,
    "hour_shrink": 8.0,
    "site_hour_shrink": 4.0,
    "smooth_texture": 0.0,
    "smooth_event": 0.0,
}

POST_CFG = {
    "temp_aves": 1.0,
    "temp_texture": 1.0,
    "top_k_event": 2,
    "top_k_texture": 4,
    "power": 1.0,
    "event_max_alpha": 0.03,
    "texture_smooth_alpha": 0.03,
    "rank_power_event": 1.0,
    "rank_power_texture": 1.0,
}

In [ ]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def safe_logit(p: np.ndarray, eps: float = 1e-4) -> np.ndarray:
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p) - np.log1p(-p)


def binary_entropy(probs: np.ndarray, eps: float = 1e-4) -> np.ndarray:
    probs = np.clip(probs, eps, 1.0 - eps)
    entropy = -(probs * np.log(probs) + (1.0 - probs) * np.log(1.0 - probs))
    entropy /= np.log(2.0)
    return entropy.astype(np.float32)


def parse_site_and_hour(filename: str) -> tuple[str, int]:
    stem = Path(filename).stem
    match = FILENAME_RE.match(stem)
    if not match:
        return "UNKNOWN", -1
    return match.group(1), int(match.group(3)[:2])


def _require_module(name: str):
    return importlib.import_module(name)


def read_soundscape_60s(path: Path) -> np.ndarray:
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr} for {path}; expected {SR}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    return y[:FILE_SAMPLES]


def build_group_orders(df: pd.DataFrame) -> list[np.ndarray]:
    out: list[np.ndarray] = []
    for _filename, sub_df in df.groupby("filename", sort=False):
        out.append(sub_df.sort_values("end_sec").index.to_numpy(dtype=np.int32))
    return out


def smooth_scores_for_cols(
    scores: np.ndarray,
    group_orders: list[np.ndarray],
    cols: np.ndarray,
    alpha: float,
) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0:
        return scores
    out = scores.copy()
    for group_idx in group_orders:
        x = out[np.ix_(group_idx, cols)]
        prev = np.concatenate([x[:1], x[:-1]], axis=0)
        nxt = np.concatenate([x[1:], x[-1:]], axis=0)
        out[np.ix_(group_idx, cols)] = (1.0 - alpha) * x + 0.5 * alpha * (prev + nxt)
    return out


def fit_prior_tables(meta_df: pd.DataFrame, y_true: np.ndarray) -> dict[str, Any]:
    meta_df = meta_df.reset_index(drop=True)
    global_p = y_true.mean(axis=0).astype(np.float32)

    site_to_i: dict[str, int] = {}
    site_n: list[int] = []
    site_p: list[np.ndarray] = []
    for site, idx in meta_df.groupby("site").groups.items():
        row_idx = np.asarray(list(idx), dtype=np.int32)
        site_to_i[str(site)] = len(site_n)
        site_n.append(int(len(row_idx)))
        site_p.append(y_true[row_idx].mean(axis=0).astype(np.float32))

    hour_to_i: dict[int, int] = {}
    hour_n: list[int] = []
    hour_p: list[np.ndarray] = []
    for hour, idx in meta_df.groupby("hour_utc").groups.items():
        row_idx = np.asarray(list(idx), dtype=np.int32)
        hour_to_i[int(hour)] = len(hour_n)
        hour_n.append(int(len(row_idx)))
        hour_p.append(y_true[row_idx].mean(axis=0).astype(np.float32))

    site_hour_to_i: dict[tuple[str, int], int] = {}
    site_hour_n: list[int] = []
    site_hour_p: list[np.ndarray] = []
    for (site, hour), idx in meta_df.groupby(["site", "hour_utc"]).groups.items():
        row_idx = np.asarray(list(idx), dtype=np.int32)
        site_hour_to_i[(str(site), int(hour))] = len(site_hour_n)
        site_hour_n.append(int(len(row_idx)))
        site_hour_p.append(y_true[row_idx].mean(axis=0).astype(np.float32))

    n_classes = y_true.shape[1]
    return {
        "global_p": global_p,
        "site_to_i": site_to_i,
        "site_n": np.asarray(site_n, dtype=np.float32),
        "site_p": np.stack(site_p).astype(np.float32) if site_p else np.zeros((0, n_classes), dtype=np.float32),
        "hour_to_i": hour_to_i,
        "hour_n": np.asarray(hour_n, dtype=np.float32),
        "hour_p": np.stack(hour_p).astype(np.float32) if hour_p else np.zeros((0, n_classes), dtype=np.float32),
        "site_hour_to_i": site_hour_to_i,
        "site_hour_n": np.asarray(site_hour_n, dtype=np.float32),
        "site_hour_p": np.stack(site_hour_p).astype(np.float32) if site_hour_p else np.zeros((0, n_classes), dtype=np.float32),
    }


def prior_logits(
    sites: np.ndarray,
    hours: np.ndarray,
    tables: dict[str, Any],
    site_shrink: float,
    hour_shrink: float,
    site_hour_shrink: float,
) -> np.ndarray:
    n_rows = len(sites)
    probs = np.repeat(tables["global_p"][None, :], n_rows, axis=0).astype(np.float32, copy=True)
    for row_idx, (site, hour) in enumerate(zip(sites, hours, strict=True)):
        site_key = str(site)
        hour_key = int(hour)

        hour_i = tables["hour_to_i"].get(hour_key)
        if hour_i is not None:
            n = tables["hour_n"][hour_i]
            mix = float(n / (n + hour_shrink))
            probs[row_idx] = mix * tables["hour_p"][hour_i] + (1.0 - mix) * probs[row_idx]

        site_i = tables["site_to_i"].get(site_key)
        if site_i is not None:
            n = tables["site_n"][site_i]
            mix = float(n / (n + site_shrink))
            probs[row_idx] = mix * tables["site_p"][site_i] + (1.0 - mix) * probs[row_idx]

        site_hour_i = tables["site_hour_to_i"].get((site_key, hour_key))
        if site_hour_i is not None:
            n = tables["site_hour_n"][site_hour_i]
            mix = float(n / (n + site_hour_shrink))
            probs[row_idx] = mix * tables["site_hour_p"][site_hour_i] + (1.0 - mix) * probs[row_idx]

    return safe_logit(probs).astype(np.float32)


def fuse_base_scores(
    raw_scores: np.ndarray,
    prior_scores: np.ndarray,
    class_family: dict[int, str],
    group_orders: list[np.ndarray],
    prior_weight_event: float,
    prior_weight_texture: float,
    smooth_texture: float,
    smooth_event: float,
) -> np.ndarray:
    event_cols = np.asarray(
        [cls_idx for cls_idx, family in class_family.items() if family not in TEXTURE_TAXA],
        dtype=np.int32,
    )
    texture_cols = np.asarray(
        [cls_idx for cls_idx, family in class_family.items() if family in TEXTURE_TAXA],
        dtype=np.int32,
    )
    base = raw_scores.copy()
    base[:, event_cols] = raw_scores[:, event_cols] + prior_weight_event * prior_scores[:, event_cols]
    base[:, texture_cols] = raw_scores[:, texture_cols] + prior_weight_texture * prior_scores[:, texture_cols]
    base = smooth_scores_for_cols(base, group_orders, texture_cols, smooth_texture)
    base = smooth_scores_for_cols(base, group_orders, event_cols, smooth_event)
    return base.astype(np.float32)


def build_transfer_context_features(probs: np.ndarray, group_orders: list[np.ndarray]) -> dict[str, np.ndarray]:
    prev_rows = probs.copy()
    next_rows = probs.copy()
    file_mean_rows = np.zeros_like(probs, dtype=np.float32)
    file_min_rows = np.zeros_like(probs, dtype=np.float32)
    file_max_rows = np.zeros_like(probs, dtype=np.float32)
    for group in group_orders:
        group_probs = probs[group]
        prev_rows[group] = np.concatenate([group_probs[:1], group_probs[:-1]], axis=0)
        next_rows[group] = np.concatenate([group_probs[1:], group_probs[-1:]], axis=0)
        file_mean_rows[group] = group_probs.mean(axis=0, keepdims=True)
        file_min_rows[group] = group_probs.min(axis=0, keepdims=True)
        file_max_rows[group] = group_probs.max(axis=0, keepdims=True)
    return {
        "prev": prev_rows.astype(np.float32),
        "next": next_rows.astype(np.float32),
        "file_mean": file_mean_rows.astype(np.float32),
        "file_min": file_min_rows.astype(np.float32),
        "file_max": file_max_rows.astype(np.float32),
        "file_range": np.clip(file_max_rows - file_min_rows, 0.0, 1.0).astype(np.float32),
    }


def build_global_activity_features(probs: np.ndarray, group_orders: list[np.ndarray]) -> dict[str, np.ndarray]:
    row_max = probs.max(axis=1).astype(np.float32)
    top_k = min(3, probs.shape[1])
    if top_k <= 0:
        row_top3_mean = np.zeros((len(probs),), dtype=np.float32)
    else:
        top_vals = np.partition(probs, probs.shape[1] - top_k, axis=1)[:, -top_k:]
        row_top3_mean = top_vals.mean(axis=1).astype(np.float32)

    file_row_max_mean = np.zeros((len(probs),), dtype=np.float32)
    for group in group_orders:
        file_row_max_mean[group] = np.float32(row_max[group].mean())

    return {
        "row_max": row_max,
        "row_top3_mean": row_top3_mean,
        "file_row_max_mean": file_row_max_mean,
    }


def predict_clip_student(bundle: dict[str, Any], emb: np.ndarray, raw: np.ndarray) -> np.ndarray:
    emb_scaled = bundle["emb_scaler"].transform(emb)
    z = bundle["pca"].transform(emb_scaled).astype(np.float32)
    x = np.concatenate([z, raw.astype(np.float32)], axis=1).astype(np.float32)
    x = bundle["feature_scaler"].transform(x)
    pred_logits = bundle["model"].predict(x).astype(np.float32)
    return sigmoid(pred_logits)


def fit_full_classwise_calibration(
    transfer_train: np.ndarray,
    transfer_test: np.ndarray,
    base_train: np.ndarray,
    base_test: np.ndarray,
    raw_train: np.ndarray,
    raw_test: np.ndarray,
    prior_train: np.ndarray,
    prior_test: np.ndarray,
    teacher_train: np.ndarray,
    y_train: np.ndarray,
    transfer_context_train: dict[str, np.ndarray],
    transfer_context_test: dict[str, np.ndarray],
    base_context_train: dict[str, np.ndarray],
    base_context_test: dict[str, np.ndarray],
    transfer_global_train: dict[str, np.ndarray],
    transfer_global_test: dict[str, np.ndarray],
    base_global_train: dict[str, np.ndarray],
    base_global_test: dict[str, np.ndarray],
) -> tuple[np.ndarray, dict[str, int]]:
    cfg = CLIP_CFG
    fallback_test = (
        cfg["fallback_transfer_weight"] * transfer_test
        + (1.0 - cfg["fallback_transfer_weight"]) * base_test
    ).astype(np.float32)
    final_test = fallback_test.copy()
    fitted_classes = 0

    for class_idx in range(y_train.shape[1]):
        y_col = y_train[:, class_idx]
        if int(y_col.sum()) < cfg["min_calibration_pos"] or int(y_col.sum()) == len(y_col):
            continue

        x_train_parts = [
            transfer_train[:, class_idx],
            base_train[:, class_idx],
            raw_train[:, class_idx],
            prior_train[:, class_idx],
            transfer_global_train["row_max"],
            transfer_global_train["row_top3_mean"],
            transfer_global_train["file_row_max_mean"],
            base_global_train["row_max"],
            base_global_train["row_top3_mean"],
            base_global_train["file_row_max_mean"],
            transfer_context_train["prev"][:, class_idx],
            transfer_context_train["next"][:, class_idx],
            transfer_context_train["file_mean"][:, class_idx],
            transfer_context_train["file_max"][:, class_idx],
            base_context_train["prev"][:, class_idx],
            base_context_train["next"][:, class_idx],
            base_context_train["file_mean"][:, class_idx],
            base_context_train["file_max"][:, class_idx],
        ]
        x_test_parts = [
            transfer_test[:, class_idx],
            base_test[:, class_idx],
            raw_test[:, class_idx],
            prior_test[:, class_idx],
            transfer_global_test["row_max"],
            transfer_global_test["row_top3_mean"],
            transfer_global_test["file_row_max_mean"],
            base_global_test["row_max"],
            base_global_test["row_top3_mean"],
            base_global_test["file_row_max_mean"],
            transfer_context_test["prev"][:, class_idx],
            transfer_context_test["next"][:, class_idx],
            transfer_context_test["file_mean"][:, class_idx],
            transfer_context_test["file_max"][:, class_idx],
            base_context_test["prev"][:, class_idx],
            base_context_test["next"][:, class_idx],
            base_context_test["file_mean"][:, class_idx],
            base_context_test["file_max"][:, class_idx],
        ]
        x_train = np.stack(x_train_parts, axis=1).astype(np.float32)
        x_test = np.stack(x_test_parts, axis=1).astype(np.float32)

        teacher_conf = 1.0 - binary_entropy(teacher_train[:, class_idx])
        row_teacher_weight = cfg["soundscape_min_teacher_weight"] + (
            cfg["soundscape_target_teacher_weight"] - cfg["soundscape_min_teacher_weight"]
        ) * teacher_conf
        target_probs = np.clip(
            row_teacher_weight * teacher_train[:, class_idx]
            + (1.0 - row_teacher_weight) * y_col.astype(np.float32),
            1e-4,
            1.0 - 1e-4,
        )

        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(x_train)
        x_test_scaled = scaler.transform(x_test)

        model = Ridge(alpha=cfg["soundscape_calib_alpha"])
        model.fit(x_train_scaled, safe_logit(target_probs))
        pred_test = sigmoid(model.predict(x_test_scaled).astype(np.float32))

        final_test[:, class_idx] = (
            (1.0 - cfg["calibration_blend"]) * fallback_test[:, class_idx]
            + cfg["calibration_blend"] * pred_test
        ).astype(np.float32)
        fitted_classes += 1

    return final_test.astype(np.float32), {
        "fitted_classes": int(fitted_classes),
        "skipped_classes": int(y_train.shape[1] - fitted_classes),
    }


def file_level_confidence_scale_masked(
    preds: np.ndarray,
    file_ids: np.ndarray,
    cols: np.ndarray,
    top_k: int,
) -> np.ndarray:
    if top_k <= 0 or len(cols) == 0:
        return preds
    out = preds.copy()
    for file_id in np.unique(file_ids):
        idx = file_ids == file_id
        view = out[np.ix_(idx, cols)]
        if len(view) == 0:
            continue
        k = min(top_k, len(view))
        sorted_view = np.sort(view, axis=0)
        topk_mean = sorted_view[-k:].mean(axis=0, keepdims=True)
        out[np.ix_(idx, cols)] = view * topk_mean
    return np.clip(out, 0.0, 1.0)


def file_level_rank_scale_masked(
    preds: np.ndarray,
    file_ids: np.ndarray,
    cols: np.ndarray,
    power: float,
) -> np.ndarray:
    if abs(power - 1.0) <= 1e-9 or len(cols) == 0:
        return preds
    out = preds.copy()
    for file_id in np.unique(file_ids):
        idx = file_ids == file_id
        view = out[np.ix_(idx, cols)]
        if len(view) == 0:
            continue
        file_max = view.max(axis=0, keepdims=True)
        scale = np.power(np.clip(file_max, 0.0, 1.0), power)
        out[np.ix_(idx, cols)] = view * scale
    return np.clip(out, 0.0, 1.0)


def temporal_reduce_for_cols(
    probs: np.ndarray,
    group_orders: list[np.ndarray],
    cols: np.ndarray,
    alpha: float,
    reduce: str,
) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0:
        return probs
    out = probs.copy()
    for group_idx in group_orders:
        x = out[np.ix_(group_idx, cols)]
        prev = np.concatenate([x[:1], x[:-1]], axis=0)
        nxt = np.concatenate([x[1:], x[-1:]], axis=0)
        if reduce == "max":
            neighbor = np.maximum(np.maximum(prev, x), nxt)
        elif reduce == "mean":
            neighbor = (prev + x + nxt) / 3.0
        else:
            raise ValueError(f"unknown reduce mode: {reduce}")
        out[np.ix_(group_idx, cols)] = (1.0 - alpha) * x + alpha * neighbor
    return np.clip(out, 0.0, 1.0)


def apply_postprocess(
    logits: np.ndarray,
    file_ids: np.ndarray,
    group_orders: list[np.ndarray],
    is_texture: np.ndarray,
) -> np.ndarray:
    probs = sigmoid(
        logits
        / np.where(is_texture, POST_CFG["temp_texture"], POST_CFG["temp_aves"]).astype(np.float32)[None, :]
    )
    event_cols = np.flatnonzero(~is_texture)
    texture_cols = np.flatnonzero(is_texture)
    probs = temporal_reduce_for_cols(
        probs, group_orders, event_cols, POST_CFG["event_max_alpha"], reduce="max"
    )
    probs = temporal_reduce_for_cols(
        probs, group_orders, texture_cols, POST_CFG["texture_smooth_alpha"], reduce="mean"
    )
    probs = file_level_confidence_scale_masked(probs, file_ids, event_cols, POST_CFG["top_k_event"])
    probs = file_level_confidence_scale_masked(probs, file_ids, texture_cols, POST_CFG["top_k_texture"])
    probs = file_level_rank_scale_masked(
        probs, file_ids, event_cols, POST_CFG["rank_power_event"]
    )
    probs = file_level_rank_scale_masked(
        probs, file_ids, texture_cols, POST_CFG["rank_power_texture"]
    )
    if abs(POST_CFG["power"] - 1.0) > 1e-9:
        probs = np.power(np.clip(probs, 0.0, 1.0), POST_CFG["power"])
    return safe_logit(probs).astype(np.float32)


def infer_perch_with_embeddings(
    paths: list[Path],
    inferencer: Any,
    primary_labels: list[str],
    class_to_bc: dict[str, Any],
    batch_files: int,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids: list[str] = []
    filenames: list[str] = []
    end_secs: list[int] = []
    sites: list[str] = []
    hours: list[int] = []
    raw_blocks: list[np.ndarray] = []
    emb_blocks: list[np.ndarray] = []

    for start in range(0, n_files, batch_files):
        batch_paths = paths[start : start + batch_files]
        batch_audio = np.empty((len(batch_paths) * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)

        for file_offset, path in enumerate(batch_paths):
            audio = read_soundscape_60s(path)
            site, hour = parse_site_and_hour(path.name)
            for win in range(N_WINDOWS):
                row_index = file_offset * N_WINDOWS + win
                end_sec = WINDOW_SEC * (win + 1)
                batch_audio[row_index] = audio[win * WINDOW_SAMPLES : (win + 1) * WINDOW_SAMPLES]
                row_ids.append(f"{path.stem}_{end_sec}")
                filenames.append(path.name)
                end_secs.append(end_sec)
                sites.append(site)
                hours.append(hour)

        bc_logits, emb = inferencer(batch_audio)

        raw = np.zeros((len(batch_audio), len(primary_labels)), dtype=np.float32)
        for class_idx, label in enumerate(primary_labels):
            bc_idx = class_to_bc.get(label)
            if bc_idx is None or not np.isfinite(bc_idx):
                continue
            bc_idx_int = int(bc_idx)
            if 0 <= bc_idx_int < bc_logits.shape[1]:
                raw[:, class_idx] = bc_logits[:, bc_idx_int]

        raw_blocks.append(raw)
        emb_blocks.append(emb)
        del batch_audio, bc_logits, emb, raw
        gc.collect()

    meta = pd.DataFrame(
        {
            "row_id": row_ids,
            "filename": filenames,
            "end_sec": end_secs,
            "site": sites,
            "hour_utc": hours,
        }
    )
    raw_full = np.concatenate(raw_blocks, axis=0).astype(np.float32)
    emb_full = np.concatenate(emb_blocks, axis=0).astype(np.float32)
    assert len(meta) == n_rows
    return meta, raw_full, emb_full


def resolve_paths() -> dict[str, Path]:
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        script_dir = Path(__file__).resolve().parent
        asset = script_dir if (script_dir / "clip_student_bundle.pkl").exists() else None
        if asset is None:
            candidates = [
                Path("/kaggle/input/birdclef-2026-cvlb-assets-0911"),
                Path("/kaggle/input/datasets/brucewu1200/birdclef-2026-cvlb-assets-0911"),
            ]
            for candidate in candidates:
                if (candidate / "clip_student_bundle.pkl").exists():
                    asset = candidate
                    break
        if asset is None:
            raise FileNotFoundError("Unable to locate Kaggle asset directory with clip_student_bundle.pkl")
        return {
            "base": Path("/kaggle/input/competitions/birdclef-2026"),
            "asset": asset,
            "model": Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"),
            "onnx": asset / "perch_v2_no_dft.onnx",
            "work": Path("/kaggle/working"),
        }
    root = Path(__file__).resolve().parents[3]
    return {
        "base": root,
        "asset": root / "tmp" / "kaggle_datasets" / "cvlb_assets_0911",
        "model": root / "tmp" / "perch_model",
        "onnx": root / "tmp" / "perch_v2_no_dft.onnx",
        "work": root / "tmp" / "kaggle_live" / "cvlb_0911_check" / "_local_out",
    }


def build_perch_inferencer(model_dir: Path, onnx_path: Path | None, *, require_onnx: bool):
    if onnx_path is not None and onnx_path.exists():
        try:
            ort = _require_module("onnxruntime")
            session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
            output_names = [output.name for output in session.get_outputs()]

            def _infer(batch_audio: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
                outputs = session.run(output_names, {"inputs": batch_audio.astype(np.float32, copy=False)})
                values = {name: value for name, value in zip(output_names, outputs, strict=True)}
                return values["label"].astype(np.float32, copy=False), values["embedding"].astype(
                    np.float32, copy=False
                )

            print("[setup] using ONNXRuntime CPU for Perch")
            return _infer, "onnxruntime"
        except Exception as exc:
            if require_onnx:
                raise RuntimeError(f"ONNXRuntime required but unavailable: {exc}") from exc
            print(f"[setup] ONNXRuntime unavailable, falling back to TensorFlow: {exc}")

    if require_onnx:
        raise RuntimeError(f"ONNX model missing or unavailable at {onnx_path}")

    print("[setup] using TensorFlow SavedModel for Perch")
    tf = _require_module("tensorflow")
    birdclassifier = tf.saved_model.load(str(model_dir))
    infer_fn = birdclassifier.signatures["serving_default"]

    def _infer(batch_audio: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        outputs = infer_fn(inputs=tf.convert_to_tensor(batch_audio))
        return outputs["label"].numpy().astype(np.float32, copy=False), outputs["embedding"].numpy().astype(
            np.float32, copy=False
        )

    return _infer, "tensorflow"


def resolve_test_paths(base: Path, sample_sub: pd.DataFrame) -> tuple[list[Path], str]:
    test_dir = base / "test_soundscapes"
    is_test_env = len(sample_sub) > 10
    if is_test_env:
        test_ss_paths: list[Path] = []
        added: set[str] = set()
        for row_id in sample_sub["row_id"].astype(str).tolist():
            file_id = "_".join(row_id.split("_")[:-1])
            if file_id in added:
                continue
            added.add(file_id)
            test_ss_paths.append(test_dir / f"{file_id}.ogg")
        return test_ss_paths, "competition_test"

    debug_limit = int(os.environ.get("BC26_DEBUG_MAX_FILES", "600"))
    debug_paths = sorted((base / "train_soundscapes").glob("*.ogg"))[:debug_limit]
    return debug_paths, "debug_train_soundscapes"

In [ ]:
# ============================================================
# Main execution — adapted from submission_main.py main()
# ============================================================
started = time.perf_counter()

sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')
taxonomy = pd.read_csv(BASE_DIR / 'taxonomy.csv')
is_test_env = len(sample_sub) > 10

print(f'sample_sub rows: {len(sample_sub)}, is_test_env: {is_test_env}')

# Load pre-trained assets
with (ASSET_DIR / 'clip_student_bundle.pkl').open('rb') as handle:
    student_bundle = pickle.load(handle)
clip_bundle = student_bundle['clip_bundle']
primary_labels = list(student_bundle['primary_labels'])
class_to_bc = dict(student_bundle['class_to_bc'])

teacher_eval = pd.read_parquet(ASSET_DIR / 'teacher_eval_rows.parquet')
teacher_npz = np.load(ASSET_DIR / 'teacher_oof_predictions.npz')
teacher_soundscape = teacher_npz['oof'].astype(np.float32)
teacher_y_true = teacher_npz['y_true'].astype(np.float32)

class_name_map = taxonomy.set_index('primary_label')['class_name'].astype(str).to_dict()
class_family = {idx: class_name_map.get(label, 'Aves') for idx, label in enumerate(primary_labels)}
is_texture = np.asarray([class_family[idx] in TEXTURE_TAXA for idx in range(len(primary_labels))], dtype=bool)

print(f'Loaded {len(primary_labels)} primary labels, {len(teacher_eval)} teacher eval rows')
print(f'Texture classes: {is_texture.sum()}, Event classes: {(~is_texture).sum()}')

In [ ]:
# Build Perch inferencer (ONNX preferred, TF fallback)
inferencer, perch_backend = build_perch_inferencer(
    model_dir=MODEL_DIR or Path('/nonexistent'),
    onnx_path=ONNX_PATH,
    require_onnx=is_test_env or (MODEL_DIR is None),
)
print(f'Perch backend: {perch_backend}')

In [ ]:
# Infer on labeled soundscapes (train set for calibration)
strict_filenames = teacher_eval['filename'].astype(str).drop_duplicates().tolist()
strict_paths = [(BASE_DIR / 'train_soundscapes' / fn) for fn in strict_filenames]
print(f'[strict] files={len(strict_paths)} rows={len(teacher_eval)}')

strict_meta, strict_raw, strict_emb = infer_perch_with_embeddings(
    strict_paths, inferencer, primary_labels, class_to_bc, batch_files=12
)
strict_meta = strict_meta.set_index('row_id').loc[teacher_eval['row_id']].reset_index()
strict_lookup = pd.Series(np.arange(len(strict_meta)), index=strict_meta['row_id'])
strict_raw = strict_raw[strict_lookup[teacher_eval['row_id']].to_numpy()]
strict_emb = strict_emb[strict_lookup[teacher_eval['row_id']].to_numpy()]

print(f'strict_raw shape: {strict_raw.shape}, strict_emb shape: {strict_emb.shape}')

In [ ]:
# Build priors and features for calibration
strict_group_orders = build_group_orders(teacher_eval)
prior_tables = fit_prior_tables(teacher_eval[['filename', 'site', 'hour_utc']], teacher_y_true)

strict_prior = prior_logits(
    teacher_eval['site'].to_numpy(),
    teacher_eval['hour_utc'].to_numpy(),
    prior_tables,
    site_shrink=CLIP_CFG['site_shrink'],
    hour_shrink=CLIP_CFG['hour_shrink'],
    site_hour_shrink=CLIP_CFG['site_hour_shrink'],
)

strict_base = fuse_base_scores(
    strict_raw, strict_prior, class_family, strict_group_orders,
    prior_weight_event=CLIP_CFG['prior_weight_event'],
    prior_weight_texture=CLIP_CFG['prior_weight_texture'],
    smooth_texture=CLIP_CFG['smooth_texture'],
    smooth_event=CLIP_CFG['smooth_event'],
)

strict_transfer = predict_clip_student(clip_bundle, strict_emb, strict_raw)
strict_transfer_ctx = build_transfer_context_features(strict_transfer, strict_group_orders)
strict_base_ctx = build_transfer_context_features(strict_base, strict_group_orders)
strict_transfer_global = build_global_activity_features(strict_transfer, strict_group_orders)
strict_base_global = build_global_activity_features(strict_base, strict_group_orders)

print(f'strict_transfer shape: {strict_transfer.shape}')
print(f'Prior tables: {len(prior_tables["site_to_i"])} sites, {len(prior_tables["hour_to_i"])} hours, {len(prior_tables["site_hour_to_i"])} site×hour')

In [ ]:
# Infer on test soundscapes
test_dir = BASE_DIR / 'test_soundscapes'
if is_test_env:
    test_ss_paths = []
    added = set()
    for row_id in sample_sub['row_id'].astype(str).tolist():
        file_id = '_'.join(row_id.split('_')[:-1])
        if file_id in added:
            continue
        added.add(file_id)
        test_ss_paths.append(test_dir / f'{file_id}.ogg')
    run_mode = 'competition_test'
else:
    debug_limit = int(os.environ.get('BC26_DEBUG_MAX_FILES', '600'))
    test_ss_paths = sorted((BASE_DIR / 'train_soundscapes').glob('*.ogg'))[:debug_limit]
    run_mode = 'debug_train_soundscapes'

print(f'[test] mode={run_mode} files={len(test_ss_paths)}')

test_meta, test_raw, test_emb = infer_perch_with_embeddings(
    test_ss_paths, inferencer, primary_labels, class_to_bc, batch_files=12
)
print(f'test_raw shape: {test_raw.shape}, test_emb shape: {test_emb.shape}')

In [ ]:
# Build test features
test_group_orders = build_group_orders(test_meta)
test_prior = prior_logits(
    test_meta['site'].to_numpy(),
    test_meta['hour_utc'].to_numpy(),
    prior_tables,
    site_shrink=CLIP_CFG['site_shrink'],
    hour_shrink=CLIP_CFG['hour_shrink'],
    site_hour_shrink=CLIP_CFG['site_hour_shrink'],
)
test_base = fuse_base_scores(
    test_raw, test_prior, class_family, test_group_orders,
    prior_weight_event=CLIP_CFG['prior_weight_event'],
    prior_weight_texture=CLIP_CFG['prior_weight_texture'],
    smooth_texture=CLIP_CFG['smooth_texture'],
    smooth_event=CLIP_CFG['smooth_event'],
)
test_transfer = predict_clip_student(clip_bundle, test_emb, test_raw)
test_transfer_ctx = build_transfer_context_features(test_transfer, test_group_orders)
test_base_ctx = build_transfer_context_features(test_base, test_group_orders)
test_transfer_global = build_global_activity_features(test_transfer, test_group_orders)
test_base_global = build_global_activity_features(test_base, test_group_orders)

# Per-class Ridge calibration
final_logits, calib_info = fit_full_classwise_calibration(
    transfer_train=strict_transfer,
    transfer_test=test_transfer,
    base_train=strict_base,
    base_test=test_base,
    raw_train=strict_raw,
    raw_test=test_raw,
    prior_train=strict_prior,
    prior_test=test_prior,
    teacher_train=teacher_soundscape,
    y_train=teacher_y_true,
    transfer_context_train=strict_transfer_ctx,
    transfer_context_test=test_transfer_ctx,
    base_context_train=strict_base_ctx,
    base_context_test=test_base_ctx,
    transfer_global_train=strict_transfer_global,
    transfer_global_test=test_transfer_global,
    base_global_train=strict_base_global,
    base_global_test=test_base_global,
)
print(f'Calibration: {calib_info}')

In [ ]:
# Post-processing
file_ids = test_meta['filename'].astype('category').cat.codes.to_numpy()
final_logits = apply_postprocess(final_logits, file_ids, test_group_orders, is_texture)
final_probs = sigmoid(final_logits)

# Build submission
prediction_df = pd.DataFrame(final_probs, columns=primary_labels)
prediction_df.insert(0, 'row_id', test_meta['row_id'].values)

if run_mode == 'competition_test':
    submission = pd.merge(sample_sub[['row_id']], prediction_df, on='row_id', how='left')
    submission[primary_labels] = submission[primary_labels].fillna(0.0).astype(np.float32)
    submission.to_csv(WORK_DIR / 'submission.csv', index=False)
else:
    prediction_df[primary_labels] = prediction_df[primary_labels].astype(np.float32)
    prediction_df.to_csv(WORK_DIR / 'submission.csv', index=False)

elapsed = time.perf_counter() - started
summary = {
    'run_mode': run_mode,
    'perch_backend': perch_backend,
    'strict_files': len(strict_paths),
    'strict_rows': len(teacher_eval),
    'test_files': len(test_ss_paths),
    'test_rows': len(test_meta),
    'fitted_classes': calib_info['fitted_classes'],
    'elapsed_sec': round(elapsed, 3),
}
print(json.dumps(summary, indent=2))
print(f'Submission saved to {WORK_DIR / "submission.csv"} ({len(prediction_df)} rows)')